In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os


def prepare():
    module_path = os.path.abspath(os.path.join('../','../'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_collective import run

In [ ]:
model_params = dict(
    label = "GCN_sparsity_2", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.001,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [ ]:
data_params = dict(
    dataset = "csbm",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 20,
        n_trn_unlabeled = 0,
        n_val = 20,
        n_test = 160,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        seed = 0 # used to generate the dataset & data split
    )
)

In [ ]:
# import pandas as pd
# import time

# seeds = [0]
# delta = 0.2
# certificate_params["delta"] = delta

# metrics = [
#     "accuracy_test",
#     "accuracy_trn",
#     "accuracy_cert_pois_robust",
#     "accuracy_cert_pois_unrobust",
# ]

# summary = []

# for seed in seeds:
#     data_params["specification"]["seed"] = seed
    
#     start_time = time.time()
#     result = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
#     end_time = time.time()
#     runtime = round(end_time - start_time, 2)
    
#     summary.append({k: result[k] for k in metrics} | {"delta": delta} | {"runtime": runtime})
    
    
# df = pd.DataFrame(summary, index=[f"Seed {s}" for s in seeds])
# df.index.name = "seed"
# df.to_csv(f'results/collective/csbm-{delta:.2f}.csv', index=True)
# df

In [ ]:
#  GCN on CSBM - Complexity Cascade Data Generation
# a dense epsilon sweep for the GCN model on the CSBM dataset
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path

module_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added '{module_path}' to sys.path to find project modules.")

from exp_labelcert_collective import run

model_params = {
    'label': 'GCN',
    'model': 'GCN',
    'normalization': 'row_normalization',
    'activation': 'relu', 
    'depth': 1,
    'regularizer': 0.001, 
    'pred_method': 'svm',
    'bias': False,
    'alpha_tol': 1e-4,
    'solver': 'qplayer'
}

data_params = {
    'dataset': 'csbm',
    'learning_setting': 'transductive',
    'specification': {
        'classes': 2,
        'n_trn_labeled': 10, 
        'n_trn_unlabeled': 0,
        'n_val': 10,
        'n_test': 180,
        'sigma': 1,
        'avg_within_class_degree': 3.16, 
        'avg_between_class_degree': 0.74, 
        'K': 1.5
    }
}

other_params = {
    'device': '0',
    'dtype': torch.float64,
    'allow_tf32': False, 
    'path_gurobi_license':'/mnt/c/Users/emiel/gurobi.lic'
}

verbosity_params = {'debug_lvl': 'warning'}


eps_coarse = np.linspace(0.00, 0.30, 16).tolist()
eps_fine = np.linspace(0.13, 0.18, 26).tolist()
epsilons = sorted(set(eps_coarse + eps_fine))

seeds = range(10)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
output_filename = "final_gcn_on_csbm_collective_data.csv"
output_path = log_dir / output_filename



print("--- Starting Final Data Generation for GCN on CSBM (Collective) ---")
print(f"Running {len(epsilons)} epsilon points across {len(seeds)} seeds...")
all_results = []
start_time_total = time.time()

for seed_val in seeds:
    print(f"\nProcessing Seed: {seed_val}/{len(seeds)-1}...")
    data_params['specification']['seed'] = int(seed_val)
    
    for i, eps in enumerate(epsilons):
        certificate_params = {'delta': float(eps)}
        
        out = run(
            data_params=data_params,
            model_params=model_params,
            certificate_params=certificate_params,
            verbosity_params=verbosity_params,
            other_params=other_params,
            seed=int(seed_val)
        )
        
        out.update({'delta': float(eps), 'seed': int(seed_val)})
        all_results.append(out)
        
        print(f"  ({i+1}/{len(epsilons)}) ε={eps:.4f} | Robust Ratio: {out.get('accuracy_cert_pois_robust', 'N/A'):.4f} | Gurobi Nodes: {out.get('gurobi_node_count', 'N/A')}")


df_final = pd.DataFrame(all_results)
df_final.to_csv(output_path, index=False)

end_time_total = time.time()
print(f"\n--- ✅ Experiment Complete ---")
print(f"Total runtime: {(end_time_total - start_time_total) / 60:.2f} minutes")
print(f"Final data for {len(df_final)} runs saved to '{output_path}'")
